In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder


In [2]:
import torchvision.transforms as T
from torch.utils.data import random_split

transforms = T.Compose([
    T.Resize((224, 224)),
    T.AutoAugment(), 
    T.ToTensor(),    
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

dataset = ImageFolder("/home/paul/Sujets ANADER/Datasets/Rubber_Leaf_Dataset/Raw_Dataset", transform = transforms)
trainset, testset = random_split(dataset, [int(0.8*len(dataset)), int(0.2*len(dataset))])
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader = DataLoader(testset, batch_size=32, shuffle=False)

In [3]:
dataset.classes

['Anthracnose', 'Dry_Leaf', 'Healthy', 'Leaf_Spot']

In [4]:
len(dataset.classes)

4

In [5]:
model = models.resnet152(pretrained = True)

for param in model.parameters():

    param.requires_grad = False

for param in model.layer4.parameters():

    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

/home/paul/Sujets ANADER/Rubber tree/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/paul/Sujets ANADER/Rubber tree/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001)

In [7]:
epochs = 15

for epoch in range(epochs):

    model.train()

    total_loss = 0

    img_num = 0

    for img, label in trainloader:

        img = img.to(device)

        label = label.to(device)

        output = model(img)

        loss = criterion(output, label)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()*img.size(0)

        img_num += img.size(0)

    epoch_loss = total_loss / img_num

    print(f"Epoch {epoch+1}/{epochs}, loss: {epoch_loss}")



Epoch 1/15, loss: 0.22441515985919142
Epoch 2/15, loss: 0.06697707396835603
Epoch 3/15, loss: 0.06904005517797737
Epoch 4/15, loss: 0.054262057566298066
Epoch 5/15, loss: 0.027227333999752742
Epoch 6/15, loss: 0.017029495312679752
Epoch 7/15, loss: 0.03010185465655536
Epoch 8/15, loss: 0.03458408790442493
Epoch 9/15, loss: 0.03771539983251292
Epoch 10/15, loss: 0.019401707707057257
Epoch 11/15, loss: 0.018629557715030387
Epoch 12/15, loss: 0.007344454844270466
Epoch 13/15, loss: 0.04366743009859942
Epoch 14/15, loss: 0.022433284943757428
Epoch 15/15, loss: 0.019182630825985824


In [8]:
model.eval()
correct = 0
total = 0
with torch.no_grad():

    for img, label in testloader:

        img = img.to(device)

        label = label.to(device)

        output = model(img)

        _, predicted = torch.max(output.data, 1)

        total += label.size(0)

        correct += (predicted == label).sum().item()

In [9]:
accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9943


In [10]:
pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_true = []

y_pred = [] 

with torch.no_grad():

    for img, label in testloader:

        img = img.to(device)

        label = label.to(device)

        output = model(img)

        _, predicted = torch.max(output.data, 1)

        y_true.extend(label.cpu().numpy())

        y_pred.extend(predicted.cpu().numpy())

print("Score sur le test set:", accuracy_score(y_true, y_pred))

Score sur le test set: 1.0


In [12]:
from sklearn.metrics import accuracy_score

y_true_train_ = []
y_pred_train_ = []

with torch.no_grad():
    for img, label in trainloader:
        img = img.to(device)
        label = label.to(device)
        output = model(img)
        _, predicted = torch.max(output.data, 1)
        y_true_train_.extend(label.cpu().numpy())
        y_pred_train_.extend(predicted.cpu().numpy())

train_accuracy = accuracy_score(y_true_train_, y_pred_train_)
print(f"Score sur le train: {train_accuracy:.4f}")

Score sur le train: 0.9957


In [14]:
report = classification_report(y_true, y_pred, target_names=dataset.classes)
print("Classification report:")
print(report)

confusion = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
confusion

Classification report:
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00        66
    Dry_Leaf       1.00      1.00      1.00        87
     Healthy       1.00      1.00      1.00       110
   Leaf_Spot       1.00      1.00      1.00        85

    accuracy                           1.00       348
   macro avg       1.00      1.00      1.00       348
weighted avg       1.00      1.00      1.00       348

Confusion Matrix:


array([[ 66,   0,   0,   0],
       [  0,  87,   0,   0],
       [  0,   0, 110,   0],
       [  0,   0,   0,  85]])

In [16]:
import joblib

torch.save(model.state_dict(), "rubber_resnet152.pth")
joblib.dump(transforms, "rubber_transforms.pkl")
joblib.dump(dataset.class_to_idx, "rubber_class_to_idx.pkl")
joblib.dump(dataset.classes, "rubber_classes.pkl")

['rubber_classes.pkl']